# Chapter 3 Mastering Relational Entities in Databricks
We will explore databases, tables, and views for organizing and managing structured data in Databricks.

## Understanding Relational Entities
Exploring the databases, tables, and views and their interactions with both the metastore and the storage systems.

Collections of database objects are called are schema.

### Two types of tables in a database
- Managed Tables
-- Created within its own database directory 
---- When dropped, both underlying data and metadata are deleted
- External Tables
-- Created outside the database directory (path specified by LOCATION parameter)
---- When dropped, only removes the metadata of the table, the underlying data remains

## Putting Relational Entities into Practice
Local hive_metastore unavaiable so I'll have to make do best I can with this study guide.

In [0]:
%sql
-- OK let's use our demo_workspace and default schema
USE CATALOG demo_workspace;

-- and we'll create a table called managed_default
CREATE TABLE manage_default
  (country STRING, code STRING, dial_code STRING);

-- and populate it with a record
INSERT INTO manage_default
  VALUES ('France', 'FR', '33');

### We're not specifying LOCATION here so this table is considered managed in this database

In [0]:
%sql
-- Executing DESCRIBE EXTENDED on our table will provide advanced metadata information
DESCRIBE EXTENDED manage_default;

### We can verify certain metadata in DESCRIBE EXTENDED
Type: MANAGED which obv indicates its a managed table
Note also: location where our table resides (abfss)
The provider, wheich confirms Delta Lake (delta)

### Next we'll create an external table 
We just simply add LOCATION argument

In [0]:
%sql
    
    
CREATE TABLE external_default
  (country STRING, code STRING, dial_code STRING)
  LOCATION 'abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_default';

INSERT INTO external_default
  VALUES ('France', 'FR', '33')

In [0]:
%sql
DESCRIBE EXTENDED external_default

### Dropping managed and external have different consequences
Let's drop the managed database first

In [0]:
%sql
DROP TABLE manage_default

### Dropping a managed table deletes its metadata from the metastore
It also deletes the data as well

In [0]:
%sql
SELECT * FROM managed_default

In [0]:
# verified by checking files for managed default
files = spark.sql("DESCRIBE DETAIL managed_default").select("location").collect()[0][0]
for f in files:
    print(f)

### However, if we drop an external- we get much different results
Let's drop it now.

In [0]:
%sql
DROP TABLE external_default;

In [0]:
files = dbutils.fs.ls("abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_default")

for f in files:
    print(f)

### As we can see, we can still see the files for the table, but we don't have the table in the schema
We can query it if we'd like.

In [0]:
%sql
    
SELECT * FROM delta.`abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_default`

### Moreover, we can delete the external tables files if we'd like
dbutils.fs.rm(path, <recurse, bool>) will enable us to do so

In [0]:
dbutils.fs.rm('abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_default', True)

## Working in a new Schema
Let's create a new database

In [0]:
%sql
CREATE SCHEMA new_default;

### Let's inspect the metadata
We can use our old friend DESCRIBE DATABASE EXTENDED

In [0]:
%sql
DESCRIBE DATABASE EXTENDED new_default

### Let's create some tables

In [0]:
%sql
USE DATABASE new_default;

CREATE TABLE managed_new
  (country STRING, code STRING, dial_code STRING);

INSERT INTO managed_new
VALUES ('France', 'FR', '33');

-- Create an external table
CREATE TABLE external_new_default
  (country STRING, code STRING, dial_code STRING)
LOCATION 'abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_new_default';

INSERT INTO external_new_default
VALUES ('France', 'FR', '33');


In [0]:
%sql
DESCRIBE EXTENDED managed_new;

In [0]:
%sql
    
DESCRIBE EXTENDED external_new_default;

In [0]:
%sql
DROP TABLE managed_new;
DROP TABLE external_new_default;

In [0]:
files = dbutils.fs.ls('abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_new_default')

for f in files:
    print(f)

## Working in a Custom-Location Schema
Lastly, we will create a database in a custom location.

In [0]:
%sql
CREATE SCHEMA custom
MANAGED LOCATION 'abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/schemas/custom'

In [0]:
%sql
-- let's check the location
DESCRIBE DATABASE EXTENDED custom;

### Let's make some tables


In [0]:
%sql
USE DATABASE custom;

-- create a managed table
CREATE TABLE managed_custom
  (country STRING, code STRING, dial_code STRING);

INSERT INTO managed_custom
VALUES ('France', 'FR', '33');

-- create an external table
CREATE TABLE external_custom
  (country STRING, code STRING, dial_code STRING)
LOCATION 'abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_custom';

INSERT INTO external_custom
VALUES ('France', 'FR', '33');

In [0]:
%sql
DESCRIBE EXTENDED managed_custom;

In [0]:
%sql
DESCRIBE EXTENDED external_custom;

### Ok that's enough of that, let's drop em like its hot

In [0]:
%sql
DROP TABLE managed_custom;
DROP TABLE external_custom;

In [0]:
files = dbutils.fs.ls('abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_custom')
for f in files:
    print(str(f) + '\n')

In [0]:
# and we can always remove them if we want
dbutils.fs.rm('abfss://unity-catalog-storage@dbstorageqh3nvwxzzfwa6.dfs.core.windows.net/7405605315193329/external_custom', True)

## Setting up Delta Tables
### CTAS Statements (CREATE TABLE AS SELECT)
CTAS Statements allow the creation and population of tables at the same time based on the results of a SELECT query.
`CREATE TABLE table_2 AS SELECT * FROM table_1`

Furthermore, we can perform transformations using CTAS.
`CREATE TABLE table_2 AS SELECT col_1, col_3 AS new_col_3 FROM table_1`

Even more advanced options open up to us:

`CREATE TABLE new_users`

`  COMMENT "Contains PII"`

`  PARTITIONED BY (city, birth_date)`

`  LOCATION 'path/to/location'`

`  AS SELECT id, name, email, birth_date, city FROM users`

### Create Table vs CTAS
Note briefly how CREATE TABLE allows for manual schema declaration whereas CTAS infers data schema
CTAS creates and populates data, create table does not populate with data.

### Table Constraints
#### NOT NULL
#### CHECK
`ALTER TABLE <table_name> ADD CONSTRAINT <constraint_name> <constraint_detail>`

Existing data must adhere to this new constraint, otherwise it will fail to create the constraint.
Once enforced, any new data that violates the constraint will result in a write failure.

### Consider
The addition of a `CHECK` constraint to a date column of a delta table:

`ALTER TABLE my_table`

`ADD CONSTRAINT valid_date CHECK (date >= '2024-01-01' AND date <= '2024-12-31');`

Check constraints resemble WHERE clauses for incoming data to be allowed into the table.
The above constraint would ensure that dates within the date colum fall within a specified range (within 2024 here). 
Any attempt to insert or update data with dates outside 2024 will be rejected.

### Cloning Delta Lake Tables
If you need to back up or duplicate Delta Lake table:
- Deep Clone
- Shallow Clone

#### Deep Clone
Copies both data and metadata from a src table to target table:

`CREATE TABLE <target_name>`

`DEEP CLONE <src_name>`

This copy process can occur incrementally, allowing you to sync changes from source to target and it will create a new table version with new changes:

`CREATE OR REPLACE TABLE <target_name>`

`DEEP CLONE <src_name>`

Deep cloning can take a long time as data is copied.

#### Shallow Clone
Quicker route, it only copies the delta transaction logs, no data copying.

`CREATE TABLE <target_name>`

`SHALLOW CLONE <src_name>`

Ideal for testing scenarios without altering the current table's data. Good for AGILE development.

##### Data integrity
Any modifications made to either clone are tracked and stored separately from the source.

## Exploring Views
### Virtualized tables without data (saved SQL query)

In [0]:
%sql
USE CATALOG demo_workspace;
-- let's use default schema once again and create a table called cars
USE SCHEMA default;

CREATE TABLE cars
(id INT, brand STRING, model STRING, year INT, price FLOAT);

INSERT INTO cars
VALUES (1, 'Tesla', 'Cybertruck', 2022, 70000),
       (2, 'Tesla', 'Model S', 2022, 90000),
       (3, 'Tesla', 'Model 3', 2022, 50000),
       (4, 'Tesla', 'Model X', 2022, 80000),
       (5, 'Mercedes-Benz', 'G-Class AMG', 2022, 150000),
       (6, 'Mercedes-Benz', 'E-Class E200', 2022, 50000),
       (7, 'Mercedes-Benz', 'GLE 450', 2022, 80000),
       (8, 'Ford', 'Fiesta ST', 2022, 25000),
       (9, 'Ford', 'Mustang Shelby GT500', 2022, 120000),
       (10, 'Ford', 'Mustang Mach-E', 2022, 90000)

In [0]:
%sql
-- Let's verify the table creation 
SHOW TABLES

### View Types
- Stored
- Temporary 
- Global Temporary

#### Stored
Referred simply as *views* similar to traditional views:

`CREATE VIEW <view_name>`

`AS <query>`

In [0]:
%sql
-- Let's create a view for only Tesla brand cars
CREATE VIEW vw_tesla_cars
AS SELECT *
   FROM cars
   WHERE brand = 'Tesla';

In [0]:
%sql
SHOW TABLES

In [0]:
%sql
SELECT * FROM vw_tesla_cars

#### Temporary Views
Views that are bound to the spark session and are automatically dropped when the session ends.
Handy for temporary data manipulations or analyses. 

`CREATE TEMP VIEW <view_name> AS <query>`

In [0]:
%sql
-- Let's create a temporary view to retrieve unique car brands in our table
CREATE TEMP VIEW vw_unique_brands
AS SELECT DISTINCT brand
   FROM cars;

SELECT * FROM vw_unique_brands;

In [0]:
%sql
SHOW TABLES

We can see the isTemporary for our temp view as well as null database as it is not persisted.

Only limited to duration of the spark session:
- Opening a new notebook
- Detaching and reattaching a notebook to a cluster
- Restarting the Python interpreter due to a Python package installation
- Restarting the cluster itself

As we saw in the new session notebook, the vw_unique_brands view did not exist in the new notebook.

#### Global Temporary Views
Behavior similar to temporary views but are tied to the cluster instead of a specified session. As long as the cluster is running, any notebook attached to it can access its global temporary view. 

`CREATE GLOBAL TEMP VIEW <view_name>`

In [0]:
%sql
CREATE GLOBAL TEMP VIEW vw_global_temp_recent_cars
AS SELECT *
   FROM cars
   WHERE year >= 2022
   ORDER BY year DESC;

SELECT * FROM global_temp.vw_global_temp_recent_cars;

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
SHOW TABLES IN global_temp;

In [0]:
%sql
-- Let's drop our view
DROP VIEW vw_tesla_cars;

In [0]:
%sql
-- We can also drop temporary views in the same manner if we want to drop them before the session ends or the cluster terminates
DROP VIEW vw_unique_brands;
DROP VIEW global_temp.vw_global_temp_recent_cars;

In [0]:
%sql
SHOW TABLES

In [0]:
%sql
SHOW TABLES IN global_temp